In [3]:
import sys

sys.path.append("..")
from collections import defaultdict

import numpy as np
import pandas as pd
from matchms import Spectrum
from tqdm import tqdm

massSpecGym_path = "../data/legacy/massSpecGymData/MassSpecGym.tsv"

df = pd.read_csv(massSpecGym_path, sep="\t")

key2index = {
    "mz": 1,
    "intensity": 2,
    "smiles": 3,
    "inchikey": 4,
    "precursor_mz": 8,
    "adduct": 9,
    "instrument_type": 10
}

instrument2spectra = defaultdict(list)

for row in tqdm(df.values):
    mz = np.array(row[key2index["mz"]].split(","), dtype=np.float32)
    intensity = np.array(row[key2index["intensity"]].split(","), dtype=np.float32)
    smiles = row[key2index["smiles"]]
    inchikey = row[key2index["inchikey"]]
    precursor_mz = row[key2index["precursor_mz"]]
    adduct = row[key2index["adduct"]]

    instrument_type = row[key2index["instrument_type"]]
    s = Spectrum(
        mz=mz,
        intensities=intensity,
        metadata={
            "smiles": smiles,
            "inchikey": inchikey,
            "precursor_mz": precursor_mz,
            "adduct": adduct
        }
    )
    instrument2spectra["all"].append(s)
    if pd.isna(instrument_type):
        continue

    instrument2spectra[instrument_type].append(s)

instrument2spectra = dict(instrument2spectra)

100%|██████████| 231104/231104 [00:17<00:00, 13533.16it/s]


In [4]:
from pathlib import Path

import numpy as np

from SpecEmbedding.utils.clean import get_ref_query, get_unique_smiles

path_dir = Path("../data/legacy/massSpecGymData")
path_dir.mkdir(parents=True, exist_ok=True)
replica_suffix = "-replication-{}"

train_ref_spectra = np.load("../data/legacy/MSBert/GNPS/Orbitrap/train_ref.npy", allow_pickle=True)
train_ref_smiles = get_unique_smiles(train_ref_spectra)

for instrument_type, spectra in instrument2spectra.items():
    smiles_seq = get_unique_smiles(spectra)
    print(len(smiles_seq))
    bool_indices = np.isin(smiles_seq, train_ref_smiles)
    exclued_smiles = smiles_seq[~bool_indices]
    print(len(exclued_smiles))
    for i in range(10):
        query, reference = get_ref_query(spectra, exclued_smiles)
        np.save(path_dir.joinpath(instrument_type + "-query" + replica_suffix.format(i + 1)), query)
        np.save(path_dir.joinpath(instrument_type + "-reference" + replica_suffix.format(i + 1)), reference)

31602
30825


split query and reference set: 100%|██████████| 30825/30825 [00:00<00:00, 123573.17it/s]


23978
23232


split query and reference set: 100%|██████████| 23232/23232 [00:00<00:00, 115880.24it/s]


13522
13072


split query and reference set: 100%|██████████| 13072/13072 [00:00<00:00, 225790.14it/s]
